# Notebook 05 — evidence-linked debrief and feedback probes

**Question.** Can a debrief retain exact evidence, avoid promoting authored intents to measured student skills, and replay its next-visit branch? These are deterministic engineering probes across the [three authored cases](../../cases/), not sampled learner encounters. The shared [`build_debrief`](../../backend/app/feedback/debrief.py), [`documentation`](../../backend/app/feedback/documentation.py), [`rubric`](../../backend/app/feedback/rubric.py), [`decisions`](../../backend/app/feedback/decisions.py), and [`resolve_visit`](../../backend/app/encounter/visit.py) contracts are reused. No model is trained or faculty score awarded.

**Source trail.** [`AGENTS.md`](../../AGENTS.md), [`context.md`](../../context.md) §14, [`PRD.md`](../../PRD.md) F-01–F-10 and L-03–L-04, [Notebook 01 audit](../docs/validation/dataset_audit.json), [Notebook 02 report](../docs/validation/action_data.json), [Notebook 03 report](../docs/validation/patient_environment.json), [Notebook 04 report](../docs/validation/grounding_eval.json). A case SHA-256, version, fact source, and exact character offset accompany the trace. The Patient–Doctor corpus has no approved import license and contributes no content.

In [ ]:
import os, sys
from pathlib import Path
os.environ["LLM_ENABLED"] = "false"
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "PRD.md").exists())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "backend"))
from backend_kelsch.scripts.eval_debrief import evaluate
from backend_kelsch.app.debrief_probe.evaluation import review_queue, na_probe
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt
report, records = evaluate()
print(report["counts"])
print(report["gates"])
assert report["gates"]["evidence_linkage_and_replay"] == "PASS"

## 1. Authored fixture and the shared contract

There are six authored scenarios: Walter's basic, absent-action and low-confidence paths; Alina's mood/safety path; Eli's explanation and contradictory-action paths. Their `Intent` hits are explicit authored inputs. Nothing here can validate a classifier, learner outcome, or clinical accuracy. The shared rule-based rubric emits provisional numbers internally, which this adapter discards. Exported suggestions carry `faculty_score: null` and `review_status: pending`. Do not interpret branch names as clinical predictions.

In [ ]:
display(pd.DataFrame([{"case": r["case_id"], "scenario": r["scenario"],
    "case_version": r["case_version"], "source_sha256": r["case_source"]["sha256"][:12],
    "claims": len(r["claims"]), "next_branch": r["branch"]["taken_id"],
    "uncertainty": ", ".join(r["branch"]["uncertain_predicates"]),
    "replayed": r["branch"]["replay_match"]} for r in records]))
print("Rubric output stored as faculty scores:", [r["faculty_approved_scores"] for r in records])

## 2. Transcript and clickable evidence links

The table links each evidence assertion to a turn shown below. An exact `start:end` slice must equal `quote`; case fact references include the source path, case hash/version and authored source metadata. For a documentation numeric discrepancy, the source is the **note span** and the comparator is the complete disclosed-fact ledger; absence cannot be proven by inventing a transcript quote. Such results are pending heuristic candidates, not clinical judgements.

In [ ]:
r = next(x for x in records if x["case_id"] == "walter_greene" and x["scenario"] == "authored")
for t in r["transcript"]:
    display(Markdown(f"<a id='turn-{t['idx']}'></a>**Turn {t['idx']} · student:** {t['student_text']}  \n**Patient:** {t['reply_text']}"))
rows = []
for claim in r["claims"]:
    for ref in claim["evidence"]:
        link = f"<a href='#turn-{ref['turn']}'>turn {ref['turn']}</a> {ref['start']}:{ref['end']}“{ref['quote']}”" if ref["kind"] == "transcript_span" else (f"<a href='../../cases/{r['case_id']}.json'>case fact {ref['fact_id']}</a>" if ref["kind"] == "source_fact" else f"note {ref['field']} {ref['start']}:{ref['end']}“{ref['quote']}”")
        rows.append({"claim": claim["kind"], "label/domain": claim.get("label", claim.get("domain_key", "")), "evidence": link})
from IPython.display import HTML
display(HTML(pd.DataFrame(rows).to_html(index=False, escape=False)))
print("Case source:", r["case_source"], "audited references:", r["evidence_refs_checked"])

## 3. Documentation, missing evidence, and abstention

The shared fallback catches invented numbers and heuristically flags omitted key facts. It does not understand arbitrary clinical prose. The absent Walter path has no positive action claim; it still follows an authored branch because the decision predicate is closed over this fixture's turn window. Low-confidence decision points retain `unclear`, and conflicting antibiotic actions are explicitly flagged. Neither becomes an accepted skill label.

In [ ]:
display(pd.DataFrame([{"case": x["case_id"], "scenario": x["scenario"],
    "numeric_note_candidate": sum(c["kind"] == "documentation_unbacked_number" for c in x["claims"]),
    "missing_key_facts": sum(c["kind"] == "documentation_missing" for c in x["claims"]),
    "positive_action_claims": sum(c["kind"] == "explicit_fixture_action" for c in x["claims"]),
    "uncertain": x["branch"]["uncertain_predicates"]} for x in records]))
print("Isolated N/A fixture:", na_probe())
assert next(x for x in records if x["scenario"] == "absent")["branch"]["taken_id"] == "b_nsaid_missed"

## 4. Branch traces and reproducibility

Each v1 predicate is evaluated by the shared decision tree; the selected authored branch's `VisitChanges` is applied to v2 twice. Matching complete next-case state hashes are deterministic replay evidence. The trace below shows the path and unresolved predicates. The uncertain shared selector currently uses the boolean value of an `unclear` predicate; those paths need faculty inspection and are not validated clinical outcomes.

In [ ]:
rows = [{"scenario": x["case_id"] + ":" + x["scenario"],
    "predicates": " | ".join(d["id"] + "=" + d["result"] for d in x["decision_points"]),
    "branch": x["branch"]["taken_id"], "next_visit": x["branch"]["next_visit_id"],
    "next_state_sha": (x["branch"]["next_visit_sha256"] or "")[:12],
    "replay": x["branch"]["replay_match"]} for x in records]
display(pd.DataFrame(rows))
fig, ax = plt.subplots(figsize=(9, 3.5))
labels = [x["case_id"].split("_")[0] + ":" + x["scenario"] for x in records]
counts = [sum(c["kind"] == "rubric_suggestion" for c in x["claims"]) for x in records]
colors = ["#d57546" if x["branch"]["uncertain_predicates"] else "#2b7d69" for x in records]
ax.bar(labels, counts, color=colors); ax.set(ylabel="Anchored suggestions", title="Fixture suggestion coverage (orange = branch review)")
ax.tick_params(axis="x", rotation=30); fig.tight_layout(); plt.show()

## 5. Safety review and faculty queue

Rubric domains 6/7 at a provisional rule-based trigger value of 1 create **review flags**, never grades. Domain 6 can be `N/A` in an isolated empty-signals fixture. The review queue exports each claim with its source-locked case hash/version and a blank expected-support label. Independent faculty can approve or reject every claim with reviewer identity and timestamp. Partial approval still yields null aggregate feedback accuracy. Confirm reviewer independence outside software. Notebook 02's trained extractor and Notebook 04's clinical disclosure, semantic verifier and difficulty gates remain **NOT EVALUABLE**.

In [ ]:
display(pd.DataFrame([{"case": x["case_id"], "scenario": x["scenario"],
    "flag_domains": x["safety_review_domain_ids"], "faculty_scores": x["faculty_approved_scores"]} for x in records]))
queue = review_queue(records)
print("Pending reviews:", len(queue), "Example:", {k: v for k, v in queue[0].items() if k != "claim"})
print("Reviewed:", report["reviews"])
print("Progression:", report["gates"])
assert report["reviews"]["feedback_accuracy"] is None

## Progression and limits

**PASS** for this engineering stage requires all exported claims to have validated offsets/fact sources, no unanchored suggestion, complete deterministic v2 replay for applicable cases, and the N/A probe to return `N/A`. **FAIL** means any failed evidence or replay assertion. **NOT EVALUABLE** for faculty feedback accuracy or rubric scoring until independent reviews of claims and domain scores exist; Notebook 02/04 trained extraction, clinical release/semantic checks, difficulty calibration and student learning retain their prior NOT EVALUABLE gates.

The note check cannot identify unsupported nonnumeric prose; the authored case facts are not independently adjudicated clinical facts. These scripts are not observed students. The shared HTTP API tests are excluded from the local non-HTTP suite because their TestClient previously stalled; CI should run them in a normal environment. See the [machine-readable report](../docs/validation/debrief_eval.json), [readable validation report](../docs/validation/debrief_eval.md), and [Notebook 06 prompt](../docs/notebook_06_prompt.txt).